# Tome 11, Chapter 3: Continuous martingales (Python companion)

Exercises first, solutions at the end. The quadratic variation of a stopped Brownian motion read off one path, exit problems solved by optional stopping, the strange life of the exponential martingale, the maximum of a drifting walk and the bias of discrete monitoring, the doubling strategy, and a computer science application: **anytime-valid A/B testing**, where Ville's inequality lets an experimenter look at the data as often as they like. Requires `numpy`.

In [ ]:
import math
import numpy as np
rng = np.random.default_rng(1103)

def bm(T, N, m=1):
    # m Brownian paths on [0, T] at N + 1 equally spaced times
    steps = rng.standard_normal((m, N)) * math.sqrt(T / N)
    return np.concatenate([np.zeros((m, 1)), np.cumsum(steps, axis=1)], axis=1)

## Exercises

**P3.1 (★) Reading the clock off one path.** Let $M$ be Brownian motion stopped when it leaves $]-0.8,0.8[$, on $[0,1.5]$, simulated with $2^{16}$ steps. Compute the dyadic sums $Q_n=\sum(\Delta M)^2$ of levels $n=2,4,\dots,16$ and compare with $\langle M\rangle_{1.5}=\tau\wedge1.5$ (Theorem 3.14, Proposition 3.15). Over $200$ paths, how does the root mean square error decrease with $n$? Compare with the rate $\sqrt{2\sum(\Delta t)^2}$ that holds for unstopped Brownian motion.

**P3.2 (★) The exit problem.** For $a=1$, $b=2$, estimate $P(B_\tau=b)$ and $\mathbb E\tau$ for the exit time of $]-a,b[$ and compare with $\frac a{a+b}$ and $ab$ (Exercise 3.2). For the exit of $]-1,1[$, compare the empirical $\mathbb Ee^{-\mu\tau}$ with $1/\cosh\sqrt{2\mu}$ for $\mu=0.5,1,2$ (Exercise 3.3).

**P3.3 (★) Where has the mass gone?** For $\mathcal E_t=e^{B_t-t/2}$ (Proposition 3.19), compute for $t=1,4,8,16$ the median, the probability $P(\mathcal E_t<0.01)$ and the empirical mean of $10^5$ samples. Using $\operatorname{Var}\mathcal E_t=e^t-1$, how many samples would you need for the empirical mean at $t=16$ to be within $10\%$ of $1$?

**P3.4 (★★) The maximum of a drifting walk, and the bias of looking at discrete times.** For $\mu=1$, simulate $\sup_{t\le20}(B_t-\mu t)$ on a grid of step $\delta$ for $\delta=0.1,0.01,0.001$, and compare its mean with $\frac1{2\mu}$ (Corollary 3.21). The discrete maximum underestimates; Broadie, Glasserman and Kou (1997) showed that adding $\beta\sqrt\delta$ with $\beta=-\zeta(\frac12)/\sqrt{2\pi}\approx0.5826$ corrects the bias to first order. Check it.

**P3.5 (★) The doubling strategy.** A gambler bets $1$ on a fair coin, doubles after each loss and stops at the first win, but has a purse of $2^{10}-1=1023$ coins, so can survive at most $10$ losses in a row. Simulate $10^5$ evenings. What is the probability of going home with one more coin? What is the mean gain? Explain with Theorem 3.6 (in its discrete form).

**P3.6 (★★★) CS application: peeking at an A/B test.** Observations $X_1,X_2,\dots$ are i.i.d. $N(\theta,1)$ (the difference between two versions of a web page, say), and we test $\theta=0$ at level $\alpha=0.05$.
(a) *Peeking.* After each observation, compute $Z_n=S_n/\sqrt n$ and stop as soon as $|Z_n|>1.96$. Under $\theta=0$, estimate the probability of a false rejection within $n\le1000$ observations. (Tome 10's law of the iterated logarithm says this tends to $1$ as the horizon grows.)
(b) *Ville's test.* Robbins' mixture martingale $M_n=\sqrt{\frac{\rho}{\rho+n}}\exp\bigl(\frac{S_n^2}{2(\rho+n)}\bigr)$ is, under $\theta=0$, a non-negative martingale with $M_0=1$ (it is the average of the likelihood ratios $e^{\theta S_n-n\theta^2/2}$ over $\theta\sim N(0,1/\rho)$). Reject as soon as $M_n\ge1/\alpha$. By Ville's inequality (Tome 10, Theorem 9.15), the probability of ever rejecting is at most $\alpha$, however long we watch. Check it, with $\rho=0.01$.
(c) Under $\theta=0.2$, compare the number of observations Ville's test needs with the fixed-sample test that looks only at $n=200$.

## Solutions

### P3.1
**How to find it.** Stop the path at the first index where $|B|\ge0.8$ and freeze it. The dyadic sum of level $n$ uses every $2^{16-n}$-th point. For an unstopped Brownian motion on $[0,T]$ with $2^n$ equal steps, $\operatorname{Var}Q_n=2\sum(\Delta t)^2=2T^2/2^n$, so the error should halve when $n$ increases by $2$.

In [ ]:
T, n = 1.5, 16
N = 2**n
def stopped(B):
    out = np.where(np.abs(B) >= 0.8)[0]
    M = B.copy(); tau = T
    if len(out):
        k = out[0]; M[k:] = B[k]; tau = k * T / N
    return M, tau

B = bm(T, N)[0]
M, tau = stopped(B)
print(f"tau ^ 1.5 = {tau:.4f}")
for lev in range(2, 17, 2):
    Q = np.sum(np.diff(M[::2**(n - lev)])**2)
    print(f"level {lev:2d}: Q = {Q:.4f}")
errs = {lev: [] for lev in range(2, 17, 2)}
paths = bm(T, N, 200)
for B in paths:
    M, tau = stopped(B)
    for lev in errs:
        errs[lev].append(np.sum(np.diff(M[::2**(n - lev)])**2) - tau)
for lev in errs:
    print(f"level {lev:2d}: rms error {math.sqrt(np.mean(np.square(errs[lev]))):.4f}   unstopped rate {math.sqrt(2*T**2/2**lev):.4f}")

The stopped martingale has a smaller error than the unstopped rate: its clock runs only until $\tau$, so effectively $T$ is replaced by $\tau\wedge T$. The error halves every two levels, as the variance computation predicts.

### P3.2
**How to find it.** Simulate on a fine grid and record the first exit. The grid overshoots the barrier a little and detects the exit late, which biases $\mathbb E\tau$ upward and $\mathbb Ee^{-\mu\tau}$ downward, by $O(\sqrt\delta)$: visible in the third decimal below.

In [ ]:
def exit_times(a, b, m, dt=1e-3, T=40.0):
    N = int(T / dt)
    x = np.zeros(m); tau = np.full(m, np.nan); side = np.zeros(m)
    alive = np.ones(m, bool)
    for k in range(1, N + 1):
        idx = np.where(alive)[0]
        if len(idx) == 0: break
        x[idx] += rng.standard_normal(len(idx)) * math.sqrt(dt)
        hit_b = idx[x[idx] >= b]; hit_a = idx[x[idx] <= -a]
        tau[hit_b] = k * dt; side[hit_b] = 1
        tau[hit_a] = k * dt; side[hit_a] = -1
        alive[hit_b] = False; alive[hit_a] = False
    return tau, side

tau, side = exit_times(1.0, 2.0, 4000)
print("P(exit at b) =", np.mean(side == 1), "  theory", 1/3)
print("E tau        =", np.nanmean(tau), "  theory", 2.0)
tau, side = exit_times(1.0, 1.0, 4000)
for mu in (0.5, 1, 2):
    print(f"mu = {mu}: E exp(-mu tau) = {np.mean(np.exp(-mu*tau)):.4f}   1/cosh = {1/math.cosh(math.sqrt(2*mu)):.4f}")

### P3.3
**How to find it.** $\mathcal E_t=e^{\sqrt tZ-t/2}$ with $Z$ standard Gaussian, so the median is $e^{-t/2}$ and $P(\mathcal E_t<0.01)=P\bigl(Z<(t/2+\log0.01)/\sqrt t\bigr)$. The standard error of the empirical mean of $m$ samples is $\sqrt{(e^t-1)/m}$; for it to be $0.1$ at $t=16$ we need $m\approx100(e^{16}-1)\approx9\times10^8$. And even this is optimistic: the variance itself is carried by even rarer samples, so the empirical mean is typically much smaller than $1$ unless $m$ is of order $e^{t}$ or more.

In [ ]:
Phi = lambda x: 0.5 * (1 + math.erf(x / math.sqrt(2)))
for t in (1, 4, 8, 16):
    E = np.exp(math.sqrt(t) * rng.standard_normal(10**5) - t / 2)
    print(f"t = {t:2d}: median {np.median(E):.4f} (theory {math.exp(-t/2):.4f}),"
          f" P(E < 0.01) = {np.mean(E < 0.01):.3f} (theory {Phi((t/2 + math.log(0.01))/math.sqrt(t)):.3f}), mean {E.mean():.3f}")
print("samples needed at t = 16:", f"{100*(math.exp(16)-1):.2e}")

### P3.4
**How to find it.** By Corollary 3.21 the supremum is exponential with mean $\frac1{2\mu}=0.5$. The horizon $20$ is long enough: the supremum is reached early because the drift wins. The grid misses the true maximum between grid points; the correction $\beta\sqrt\delta$ is the expected overshoot of a Gaussian random walk over a high level, and $\zeta(\frac12)\approx-1.4604$.

In [ ]:
mu, T, m = 1.0, 20.0, 2000
beta = 1.4603545088 / math.sqrt(2 * math.pi)
for dt in (0.1, 0.01, 0.001):
    N = int(T / dt)
    sups = np.empty(m)
    for j in range(0, m, 200):
        X = np.cumsum(rng.standard_normal((200, N)) * math.sqrt(dt) - mu * dt, axis=1)
        sups[j:j+200] = np.maximum(X.max(axis=1), 0)
    print(f"dt = {dt:6.3f}: mean sup {sups.mean():.4f}, corrected {sups.mean() + beta*math.sqrt(dt):.4f}  (theory 0.5)")

The raw means are too small by roughly $0.58\sqrt\delta$, that is about $0.18$, $0.06$ and $0.02$, and the corrected means agree with $0.5$ up to the Monte Carlo error, about $0.01$ with $2000$ paths. The correction is exact only to first order in $\sqrt\delta$, but it already removes almost all of the bias at $\delta=0.1$. This is the continuity correction used to price barrier options monitored once a day.

### P3.5
**How to find it.** The gambler loses everything ($1+2+\dots+512=1023$ coins) exactly when the first $10$ tosses are all losses, with probability $2^{-10}$; otherwise wins $1$ coin. The mean gain is $(1-2^{-10})\cdot1-2^{-10}\cdot1023=0$: the capital is a martingale stopped at a bounded time (at most $10$ bets), and optional stopping forbids any profit.

In [ ]:
m = 10**5
losses_before_win = rng.geometric(0.5, m) - 1     # number of losses before the first win
gain = np.where(losses_before_win >= 10, -1023, 1)
print("P(win one coin) =", np.mean(gain == 1), "  theory", 1 - 2**-10)
print("mean gain       =", gain.mean(), "  (standard error", round(gain.std()/math.sqrt(m), 3), ")")

### P3.6
**How to find it.** (a) Under the null, $Z_n$ is a Gaussian random walk normalised at each time; checking it at $1000$ times is like running $1000$ correlated tests, and the false alarm rate is far above $5\%$. (b) $M_n$ is the mixture $\int e^{\theta S_n-n\theta^2/2}\,N(0,1/\rho)(\mathrm d\theta)$; each integrand is a martingale of mean $1$ under the null (Tome 3's discrete exponential martingale), and Fubini makes the mixture one too; integrating the Gaussian gives the closed form. Ville's inequality bounds the probability of ever crossing $1/\alpha=20$ by $0.05$. (c) Under an alternative, $\log M_n\approx\frac{S_n^2}{2n}-\frac12\log\frac n\rho$ grows like $n\theta^2/2$, so the test stops after roughly $2\log(20\sqrt{n/\rho})/\theta^2$ observations.

In [ ]:
alpha, rho, n_max, m = 0.05, 0.01, 1000, 4000
def run(theta):
    X = rng.standard_normal((m, n_max)) + theta
    S = np.cumsum(X, axis=1); n = np.arange(1, n_max + 1)
    peek = (np.abs(S / np.sqrt(n)) > 1.96).any(axis=1)
    logM = 0.5 * np.log(rho / (rho + n)) + S**2 / (2 * (rho + n))
    ville = logM >= math.log(1 / alpha)
    stop = np.where(ville.any(axis=1), ville.argmax(axis=1) + 1, np.inf)
    fixed = np.abs(S[:, 199] / math.sqrt(200)) > 1.96
    return peek, stop, fixed
peek, stop, fixed = run(0.0)
print("null: peeking false alarm rate      ", peek.mean())
print("null: Ville test false alarm rate   ", np.mean(np.isfinite(stop)), " (guaranteed <= 0.05 forever)")
peek, stop, fixed = run(0.2)
print("theta = 0.2: Ville test power within 1000 obs", np.mean(np.isfinite(stop)),
      "  median stopping time", np.median(stop))
print("theta = 0.2: fixed n = 200 test power        ", fixed.mean())
peek, stop, fixed = run(0.5)
print("theta = 0.5: Ville test median stopping time ", np.median(stop))

Peeking with the classical test rejects a true null about half of the time within $1000$ observations, and the rate keeps growing with the horizon. Ville's test keeps its promise, at every horizon at once (here it is very conservative). The price is paid under the alternative: at $\theta=0.2$ it needs a median of about $360$ observations, more than the $200$ of a fixed design tuned in advance to this effect size. In exchange it never needs to know the effect size, it may stop as soon as the evidence is overwhelming (about $50$ observations at $\theta=0.5$), and it may continue when the evidence is weak. This is the principle behind the "always valid p-values" used by online experimentation platforms (Johari, Koomen, Pekelis and Walsh, *Operations Research*, 2022) and the confidence sequences of Howard, Ramdas, McAuliffe and Sekhon (*Annals of Statistics*, 2021).